In [1]:
import io
import os
import pathlib
import subprocess

from IPython.display import display, Markdown
import polars

from utils import list_code

TRAFFIC_GENERATION_PATH = pathlib.Path.cwd() / "03_3_traffic_generation"

os.environ["TRAFFIC_GENERATION_PATH"] = str(TRAFFIC_GENERATION_PATH)

# Traffic Generation

A graphical overview of this section.

<div>
<img title="A graphical overview of this section" src="03_3_traffic_generation/method.svg" width="300"/>
</div>

## Orchestration

From our paper, Section 3.3:

> 1) _D1 (direct, single server)_ [...]: One shared server that serves both DNS and non-DNS data. The client communicates to directly with the server.
> 2) _P1 (via proxy, single server)_ [...]: One shared server that serves both DNS and non-DNS data. The client communicates to the server via proxy.
> 3) _D2 (direct, split server)_ [...]: Two servers that serve DNS and non-DNS data. The client communicates to both servers directly.
> 4) _P2 (split server, via proxy)_ [...]: Two servers that serve DNS and non-DNS data. The client communicates to both servers via a proxy.

We use `docker compose` to orchestrate our traffic generation. You can find the setups in the `03_3_traffic_generation/docker-compose*.yaml` files.

- HTTPS scenarios with unconstrained link layer
  + D1: [`03_3_traffic_generation/docker-compose-http-d1.yaml`](./03_3_traffic_generation/docker-compose-http-d1.yaml)
  + P1: [`03_3_traffic_generation/docker-compose-http-p1.yaml`](./03_3_traffic_generation/docker-compose-http-p1.yaml)
  + D2: [`03_3_traffic_generation/docker-compose-http-d2.yaml`](./03_3_traffic_generation/docker-compose-http-d2.yaml)
  + P2: [`03_3_traffic_generation/docker-compose-http-p2.yaml`](./03_3_traffic_generation/docker-compose-http-p2.yaml)
- CoAP scenarios (including unencryptet CoAP, CoAPS, OSCORE, and Onion OSCORE) with unconstrained link layer.
  + D1: [`03_3_traffic_generation/docker-compose-coap-d1.yaml`](./03_3_traffic_generation/docker-compose-coap-d1.yaml)
  + P1: [`03_3_traffic_generation/docker-compose-coap-p1.yaml`](./03_3_traffic_generation/docker-compose-coap-p1.yaml)
  + D2: [`03_3_traffic_generation/docker-compose-coap-d2.yaml`](./03_3_traffic_generation/docker-compose-coap-d2.yaml)
  + P2: [`03_3_traffic_generation/docker-compose-coap-p2.yaml`](./03_3_traffic_generation/docker-compose-coap-p2.yaml)
- CoAP scenarios (including CoAPS, OSCORE, and Onion OSCORE) with constrained link layer (SCHC).
  + D1: [`03_3_traffic_generation/docker-compose-coap-schc-d1.yaml`](./03_3_traffic_generation/docker-compose-coap-schc-d1.yaml)
  + P1: [`03_3_traffic_generation/docker-compose-coap-schc-p1.yaml`](./03_3_traffic_generation/docker-compose-coap-schc-p1.yaml)
  + D2: [`03_3_traffic_generation/docker-compose-coap-schc-d2.yaml`](./03_3_traffic_generation/docker-compose-coap-schc-d2.yaml)
  + P2: [`03_3_traffic_generation/docker-compose-coap-schc-p2.yaml`](./03_3_traffic_generation/docker-compose-coap-schc-p2.yaml)

In general overview of the structure of these files you can find in the figure below. Services and networks dependent on scenarios are grayed out, all other services and networks are always used. Please have a look into the respective YAML files and scripts for more details. Note that the database in this case is a PostgreSQL database, which we convert in `database/init_database.sh` from the `03_2_data_collection/2024-09-01-sample.sqlite3` SQLite database generated in [`03_2_data_collection.ipynb`](../03_2_data_collection.ipynb).

<div>
<img title="A graphical overview of the docker-compose files" src="03_3_traffic_generation/docker-compose.svg" width="800"/>
</div>

The containers of the `docker compose` projects are based on the following images (see [README.md](../README.md) for alternative repository versions):

- `coap-*`: `miri64/eurosp26-coap`
- `http-*`: `miri64/eurosp26-http`
- `sniffer-*`: `miri64/eurosp26-sniffer`
- `db`: `miri64/eurosp26-database`

The different SCHC rules for the [`03_3_traffic_generation/schc/schc.py`](./03_3_traffic_generation/schc/schc.py) script can be found in the respective JSON files for [OpenSCHC](https://openschc.github.io/openschc/) in `03_3_traffic_generation/schc/`. The table below orders maps them to their respective scenario group.

| Topology | Rule Set | CoAPS | OSCORE | Onion OSCORE |
|----------|----------|-------|--------|--------------|
| **D1** | _split_ | [`coaps-schc-d1-rules.json`](03_3_traffic_generation/schc/coaps-schc-d1-rules.json) | [`oscore-base-schc-d1-rules.json`](03_3_traffic_generation/schc/oscore-base-schc-d1-rules.json) | [`oscore-schc-d1-rules.json`](03_3_traffic_generation/schc/oscore-schc-d1-rules.json) |
| **P1** | _split_ | [`coaps-schc-p1-rules.json`](03_3_traffic_generation/schc/coaps-schc-p1-rules.json) | [`oscore-base-schc-p1-rules.json`](03_3_traffic_generation/schc/oscore-base-schc-p1-rules.json) | [`oscore-schc-p1-rules.json`](03_3_traffic_generation/schc/oscore-schc-p1-rules.json) |
| **D2** | _split_ | [`coaps-schc-d2-rules.json`](03_3_traffic_generation/schc/coaps-schc-d2-rules.json) | [`oscore-base-schc-d2-rules.json`](03_3_traffic_generation/schc/oscore-base-schc-d2-rules.json) | [`oscore-schc-d2-rules.json`](03_3_traffic_generation/schc/oscore-schc-d2-rules.json) |
| **D2** | _min_ | [`coaps-schc-d2-rules-min.json`](03_3_traffic_generation/schc/coaps-schc-d2-rules-min.json) | [`oscore-base-schc-d2-rules-min.json`](03_3_traffic_generation/schc/oscore-base-schc-d2-rules-min.json) | [`oscore-schc-d2-rules-min.json`](03_3_traffic_generation/schc/oscore-schc-d2-rules-min.json) |
| **D2** | _peer_ (data server) | [`coaps-schc-d2-rules-coap-server.json`](03_3_traffic_generation/schc/coaps-schc-d2-rules-coap-server.json) | [`oscore-base-schc-d2-rules-coap-server.json`](03_3_traffic_generation/schc/oscore-base-schc-d2-rules-coap-server.json) | [`oscore-schc-d2-rules-coap-server.json`](03_3_traffic_generation/schc/oscore-schc-d2-rules-coap-server.json) |
| **D2** | _peer_ (DNS server) | [`coaps-schc-d2-rules-dns-server.json`](03_3_traffic_generation/schc/coaps-schc-d2-rules-dns-server.json) | [`oscore-base-schc-d2-rules-dns-server.json`](03_3_traffic_generation/schc/oscore-base-schc-d2-rules-dns-server.json) | [`oscore-schc-d2-rules-dns-server.json`](03_3_traffic_generation/schc/oscore-schc-d2-rules-dns-server.json) |
| **P2** | _split_ | [`coaps-schc-p2-rules.json`](03_3_traffic_generation/schc/coaps-schc-p2-rules.json) | [`oscore-base-schc-p1-rules.json`](03_3_traffic_generation/schc/oscore-base-schc-p2-rules.json) | [`oscore-schc-p2-rules.json`](03_3_traffic_generation/schc/oscore-schc-p2-rules.json) |
| **P2** | _min_ | &mdash; | [`oscore-base-schc-p2-rules-min.json`](03_3_traffic_generation/schc/oscore-base-schc-p2-rules-min.json) | &mdash; |

## Run Scenarios

To let traffic generation run unassisted, we provide a script to run all scenarios. It parallelizes the runs of all network topology and link layer scenarios per protocol and format combinations, isolated by the respective Docker networks.

In [2]:
list_code(TRAFFIC_GENERATION_PATH / "run.sh")

#! /bin/bash
#
# run.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR=$( cd -- "$( dirname -- "$(realpath "$0")" )" &> /dev/null && pwd )

if [ $# -eq 1 ]; then
    case $1 in
      coap|coaps|oscore|oscore-base|https) PROTOCOL="$1" ;;
      --build) ;;
      *)  echo "usage: $0 [coap|coaps|oscore|oscore-base|https]"; exit 1 ;;
    esac
fi

export HOST_GID=$(id -g)
export HOST_UID=$(id -u)

git -C "${SCRIPT_DIR}/../" submodule update --init --recursive
chmod -R o+r ${SCRIPT_DIR}/database/ ${SCRIPT_DIR}/../input_dataset/
chmod o+x ${SCRIPT_DIR}/database/ ${SCRIPT_DIR}/../input_dataset/

if ! ls ${SCRIPT_DIR}/../input_dataset/*.sqlite3 &> /dev/null; then
    echo "No base database found!" >&2
    exit 1
fi

declare -A PREFIX_HINT_1_START=(["coap"]=6 ["http"]=7)

MAIN_ENV="${SCRIPT_DIR}"/.env
DATA_ENVS=(
    "${SCRIPT_DIR}"/.json.env
    "${SCRIPT_DIR}"/.cbor.env
)
DNS_ENVS=(
    "${SCRIPT_DIR}"/.dns-msg.env
    "${SCRIPT_DIR}"/.dns-cbor.env
)
if [ -z "${PROTOCOL}" ]; then
    SECURITIES=(
        "transport"
        "object"
        "object-base"
        ""
    )
    PROTOCOLS=(
        "coap"
        "http"
    )
elif [ "${PROTOCOL}" = "coap" ]; then
    SECURITIES=("")
    PROTOCOLS=("coap")
elif [ "${PROTOCOL}" = "coaps" ]; then
    SECURITIES=("transport")
    PROTOCOLS=("coap")
elif [ "${PROTOCOL}" = "oscore" ]; then
    SECURITIES=("object")
    PROTOCOLS=("coap")
elif [ "${PROTOCOL}" = "oscore-base" ]; then
    SECURITIES=("object-base")
    PROTOCOLS=("coap")
elif [ "${PROTOCOL}" = "https" ]; then
    SECURITIES=("transport")
    PROTOCOLS=("http")
fi
LINK_LAYERS=(
    ""
    "schc" 
)
LINK_LAYER_MODE=(
    ""
    "peer"
    "min"
)
NETWORK_SETUPS=("d1" "d2" "p1" "p2")
BLOCKWISE=(
    ""
    "block"
)

if id | grep -q "=0(root)" || id | grep -vq "docker"; then
    echo "Executing user must not be root and must be in the 'docker' group" >&2
    exit 1
fi

DOCKER_COMPOSE_PIDS=()

kill_docker() {
    for pid in ${DOCKER_COMPOSE_PIDS[@]}; do
        kill -SIGTERM "${pid}"
        tail --pid="${pid}" -f /dev/null
    done
    reset
    exit
}

trap kill_docker HUP TERM INT QUIT ABRT

if [ "$1" = "--build" ] || ! docker image ls | grep -q "pivot-eval/"; then
    docker system prune -f

    PREFIX_HINT_1="${PREFIX_HINT_1_START[${PROTOCOLS[0]}]}"
    for prot in "${PROTOCOLS[@]}"; do
        PREFIX_HINT_2=0
        for setup in "${NETWORK_SETUPS[@]}"; do
            for l2 in "${LINK_LAYERS[@]}"; do
                for l2_mode in "${LINK_LAYER_MODE[@]}"; do
                    ADDITIONAL_OPTS=""

                    if [[ "${l2}" = "schc" && ( -z "${sec}" || "${prot}" == "http" ) ]]; then
                        PREFIX_HINT_2=$(( PREFIX_HINT_2 + 1 ))
                        continue
                    elif [[ "${l2}" != "schc" && -n "${l2_mode}" ]]; then
                        PREFIX_HINT_2=$(( PREFIX_HINT_2 + 1 ))
                        continue
                    elif [[ "${l2}" = "schc" && -n "${l2_mode}" &&
                          "${setup}" != "d2" && ! (
                            "${sec}" = "object-base" && "${setup}" = "p2"
                          )
                    ]]; then
                        PREFIX_HINT_2=$(( PREFIX_HINT_2 + 1 ))
                        continue
                    fi

                    l2_iface=$(echo "${l2}${l2_mode}" | sed -E -e 's/-//g' -e 's/schc/0/g' -e 's/peer/1/' -e 's/min/2/')
                    if [ -n "${l2}" ]; then
                        l2_dash="-${l2}"
                        l2_name="-${l2}"
                        ADDITIONAL_OPTS="${ADDITIONAL_OPTS} --env-file "${SCRIPT_DIR}"/.${l2}.env"
                        if [ -n "${l2_mode}" ]; then
                            l2_name="${l2_name}-${l2_mode}"
                        fi
                    fi
                    if [ "${prot}" == "http" ] && [ "${sec}" == "transport" ]; then
                        ADDITIONAL_OPTS="${ADDITIONAL_OPTS} --env-file "${SCRIPT_DIR}"/.t

To speed things up, this script can be run on multiple machines (we used multiple virtual machines) and sliced by DNS protocol. For that you can use the optional `<protocol>` argument. The value of the argument can be seen in the following table.

| DNS protocol | Argument value |
|--------------|----------------|
| (Unencrypted) CoAP | `coap` |
| CoAPS | `coaps` |
| OSCORE | `oscore-base` |
| Onion OSCORE | `oscore` |
| HTTPS | `https` |

In [3]:
!"{TRAFFIC_GENERATION_PATH}/run.sh" --help

usage: /app/03_3_traffic_generation/run.sh [coap|coaps|oscore|oscore-base|https]


To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${DATA_COLLECTION_PATH}'/../.env/bin/activate;` before the `${TRAFFIC_GENERATION_PATH}/run.log'`).

In [4]:
%%bash

tmux new-session -s "traffic_generation" -d "script -efq '${TRAFFIC_GENERATION_PATH}/run.log' -c '${TRAFFIC_GENERATION_PATH}/run.sh'"

# to run only for e.g. unencrypte CoAP comment-out the line above and comment-in the line below.
# tmux new-session -s "traffic_generation" -d "script -efq '${TRAFFIC_GENERATION_PATH}/run.log' -c '${TRAFFIC_GENERATION_PATH}/run.sh coap'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "traffic_generation"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "traffic_generation" C-c
```

### Troubleshooting

#### Logs are Stuck

Usually just killing the session and restarting as shown above should be enough. If there is a larger problem (e.g., the logs not being completed, see below, after a few hours), we recommend to stop and remove all containers and networks related to traffic generation, e.g., using the following commands.

```sh
docker ps | awk 'NR > 1 && $2 ~ /\/eurosp26-/ && $NF !~ /jupyter/ {print $NF}' | xargs docker stop
docker ps -a | awk 'NR > 1 && $2 ~ /\/eurosp26-/ && $NF !~ /jupyter/ {print $NF}'| xargs docker rm
docker network prune
docker volume prune
```

#### Database (`db-1`) Container Prevents Starting of the Rest

The import takes some time which may timeout the health check. Just wait a few restarts of the system and it will eventually deploy. If it does not, after 10 restarts or so, use the commands above.

## Check Logs

With the script below you can check how complete logs generated by the docker setup are.

In [5]:
list_code(TRAFFIC_GENERATION_PATH / "count_client_logs.sh")

#! /bin/bash
#
# count_client_logs.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR=$( cd -- "$( dirname -- "$(realpath "$0")" )" &> /dev/null && pwd )
OUTPUT_DATASETS="$(readlink -f "${OUTPUT_DATASETS:-${SCRIPT_DIR}/../output_dataset}")"

for log in "${OUTPUT_DATASETS}"/*.client.log "${OUTPUT_DATASETS}"/*.client.log.gz; do
    if [[ "${log}" == *".gz" ]]; then
        CAT="zcat"
    else
        CAT="cat"
    fi
    "${CAT}" "${log}" | awk -v filename="${log}" '
        {
            timestamps[filename][$1 * 10**9] += 1
        }
        END {
            for (fn in timestamps) {
                sum = 0; min=0;
                for (ts in timestamps[fn]) {
                    if ((!min || min > ts) && ts != 0) { min = ts }
                    sum += timestamps[fn][ts]
                }
                print min, fn, length(timestamps[fn]), sum
            }
        }'
done | \
    sort -n | awk 'BEGIN {print "Requests", "Messages", "Log"} {print $3, $4, $2}'

When using our original dataset in [`input_dataset/2024-09-01-sample.csv.gz`](./input_dataset/2024-09-01-sample.csv.gz) there should be $58768 \times 2 = 117536$ requests (for DNS and data each) for all 296 scenarios.
The cell below checks the number of reuqests and displays the number of requests/response pairs as well as the number of outgoing application layer messages (may differ from the number of requests with block-wise transfer) for your convenience.

In [6]:
EXPECTED_REQUESTS = 117536 

proc = subprocess.check_output(f"{TRAFFIC_GENERATION_PATH}/count_client_logs.sh", text=True)
df = polars.read_csv(
    io.StringIO(proc),
    separator=" ",
)
df = df.with_columns(
    df["Log"].map_elements(
        lambda path: str(pathlib.Path(path).name),
        return_dtype=str,
    )
)
with polars.Config(
    tbl_rows=df.shape[0],
    fmt_str_lengths=df["Log"].str.len_chars().max()
):
    display(df)
# Also counts header of log
if (df["Requests"] == (EXPECTED_REQUESTS + 1)).all():
    display(Markdown(f"All {df.shape[0]} files contain the expected {EXPECTED_REQUESTS} requests"))
else:
    display(Markdown(f"The following logs are **incomplete**:"))
    display(df.filter(df["Requests"] != (EXPECTED_REQUESTS + 1)))

gzip: '/app/output_dataset/*.client.log.gz': No such file or directory


Requests,Messages,Log
i64,i64,str
117537,117537,"""oscore-base-d1_json_dns_message.client.log"""
117537,117537,"""oscore-base-schc-d1_json_dns_message.client.log"""
117537,117537,"""oscore-base-d2_json_dns_message.client.log"""
117537,117537,"""coap-d1_json_dns_message.client.log"""
117537,117537,"""oscore-base-p1_json_dns_message.client.log"""
117537,117537,"""oscore-base-p2_json_dns_message.client.log"""
117537,117537,"""coap-d2_json_dns_message.client.log"""
117537,117537,"""oscore-base-schc-d2_json_dns_message.client.log"""
117537,117537,"""oscore-base-schc-p2_json_dns_message.client.log"""


All 296 files contain the expected 117536 requests

## Compress Files

If the logs are complete, we can compress them and their associated PCAP files to save some space (the `count_client_log.sh` script also works for the compressed logs).

In [7]:
%%bash

PROCS=$(grep -c '^processor' /proc/cpuinfo)
OUTPUT_DATASET="${EVALUATION_DIR}/output_dataset"

if [ $PROCS -gt 64 ]; then
    # leave some resources to collegues ;-)
    PROCS=$(( (PROCS * 3) / 4))
fi

for file in "${OUTPUT_DATASET}"/*.{pcap,pcapng} "${OUTPUT_DATASET}"/*.{proxy,client}.log; do
    pigz -p"${PROCS}" "${file}"
done

## Generate Training Data

To generate the machine learning training data based on the PCAP files and hints from the logs, see the separate notebook [`03_3_traffic_generation/extract_from_pcaps.ipynb`](./03_3_traffic_generation/extract_from_pcaps.ipynb).

## Simulate CoAP Padding and Encrypted PIV

From intermediate files generated in [`03_3_traffic_generation/extract_from_pcaps.ipynb`](./03_3_traffic_generation/extract_from_pcaps.ipynb) we create a simulation of CoAP Padding and Partial IV encryption if they were implemented at time of evaluation. You can find the code for that in [`03_3_traffic_generation/simulate_padding_and_random_iv.ipynb`](./03_3_traffic_generation/simulate_padding_and_random_iv.ipynb).